## Part 2: Query Decomposition and Dynamic Filtering

Query decomposition is a necessary component of vector database querying. Multi-part queries that contain context from different clusters in the database will return objects for only some or none of the query components. A query like "show me black formal dresses that aren't in the basics section" has multiple semantic components. No single retrieval call will satisfy it reliably.

In this notebook, we build agents that decompose complex multi-part queries into discrete subqueries and generate dynamic SQL filters from natural language. This allows us to generate accurate responses to each semantic component of a complex query AND leverage dynamic filtering to obtain optimum retrieval.

We'll use the `products` table (2,500 products) from Part 1.

**Key concepts**: query decomposition, subquery generation, dynamic filtering, parallel retrieval, result unification.

## Setup

In [ ]:
# If you installed requirements via pip, do not uncomment this cell
#!pip install pydantic-ai psycopg2-binary

In [ ]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()

# Setup
DATABASE_URL = os.getenv("DATABASE_URL", "postgresql://root@localhost:26257/workshop?sslmode=disable")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("No OpenAI API key found. Update your .env file.")
else:
    openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

    def embed_text(text: str) -> list[float]:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding

    print(f"Database URL: {DATABASE_URL}")
    print(f"OpenAI API Key: {OPENAI_API_KEY[:10]}...")

## Reconnect to CockroachDB

In [ ]:
import psycopg2

conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = True

# Check our database to make sure our objects are still present
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM products")
    count = cur.fetchone()[0]
print(f"Connected. Table 'products' has {count} rows")

In [ ]:
# Check what columns are available
with conn.cursor() as cur:
    cur.execute("""
        SELECT column_name FROM information_schema.columns
        WHERE table_name = 'products' AND column_name != 'embedding'
        ORDER BY ordinal_position
    """)
    payload_fields = [row[0] for row in cur.fetchall()]
print(f"Columns: {payload_fields}")

---

## Introduction to Pydantic AI Agents

Before we build database-aware agents, let's explore what an agent actually is. At its simplest, a Pydantic AI agent is an LLM with **instructions** and optionally **tools**. We create it, give it a task, and it responds.

Let's start with something simple.

### Create a Simple Agent

This agent has one job: answer questions and keep the conversation going by asking a follow-up question.

In [ ]:
from pydantic_ai import Agent

simple_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions="""
    You are a helpful assistant that enjoys conversation.
    You always answer the user's question, then ask a relevant follow-up 
    question to keep the conversation going.
    """,
)

In [ ]:
# Let's see it in action
result = await simple_agent.run("What is the best programming language for data science?")
print(result.output)

### Try It: Create Your Own Agent

Give your agent a personality. Make it a pirate, a Shakespearean actor, a sports commentator — whatever you want. Fill in the instructions and run it.

In [ ]:
my_agent = Agent(
    model="openai:gpt-4.1-mini",
    # TODO: Give your agent some instructions. What personality should it have?
    instructions="""

    """,
)

result = await my_agent.run("What is the meaning of life?")
print(result.output)

In [ ]:
# Let's follow up and answer our agent's question
result = await simple_agent.run("I am so relieved, I though the meaning was deeper than that")
print(result.output)

### The Problem: No Memory


It has no context about our previous question. It doesn't know "which one" refers to programming languages. Each `agent.run()` call is a completely fresh conversation.

### Adding Memory

We can fix this by passing the conversation history from the previous call. Pydantic AI stores all messages (user + assistant) on the result object.

In [ ]:
# First message
result = await simple_agent.run("What is the best programming language for data science?")
print(result.output)
print("\n--- Now with memory ---\n")

In [ ]:
# Follow-up WITH conversation history — now it remembers
result = await simple_agent.run(
    "Which one would you recommend for a beginner?",
    message_history=result.all_messages()
)
print(result.output)

Now it knows we were talking about programming languages and gives a relevant recommendation.

**What we just learned:**
- An **Agent** is an LLM + instructions, that's an agent in its simpliest form.
- Each `agent.run()` is stateless by default and there is no memory between calls.
- Passing `message_history` gives the agent context from previous turns in our conversation.
- Later in Part 3, we'll build a full orchestration system that manages conversation state automatically, follows intent and will reason before it acts.

Now let's connect to our vector database and build agents that can use tools to inspect our data and generate precise database queries no matter how complex!

---

## Build a Dynamic Filtering Agent with Tools

When we created property filters previously, we had to express them programatically. Now, instead of hard-coding valid filter values, we will give the agent a **tool** that lets it inspect the collection at runtime. The agent decides which fields to look up, calls the tool to get the actual values from the database, and then builds precise filters.

This is more robust (works with any dataset) and demonstrates a core concept in the agentic AI REACT framework: **agents  gather information and reason before acting**.

In [ ]:
from pydantic_ai import Agent, Tool

ALLOWED_FIELDS = {"prod_name", "product_type_name", "product_group_name", "colour_group_name",
                  "department_name", "section_name", "index_name", "garment_group_name"}

# This is the tool the agent will use to inspect the table
def get_unique_values(field_name: str) -> str:
    """Look up the unique values for a column in the products table.
    Use this to find the exact values available for filtering before creating filters.
    For example, call this with 'colour_group_name' to see all available colors,
    or 'index_name' to see all available categories."""
    if field_name not in ALLOWED_FIELDS:
        return f"Error: '{field_name}' is not a valid filterable field. Use one of: {sorted(ALLOWED_FIELDS)}"
    with conn.cursor() as cur:
        cur.execute(f"SELECT DISTINCT {field_name} FROM products ORDER BY {field_name}")
        values = [row[0] for row in cur.fetchall()]
    return f"Column '{field_name}' has these values: {values}"

# Let's test the tool for ourselves to see what the agent sees
print(get_unique_values("index_name"))
print()
print(get_unique_values("colour_group_name"))

In [ ]:
# Creating our filter agent — same pattern as before, but now with a tool
smart_filter_agent = Agent(
    model="openai:gpt-4.1-mini",
    # These instructions tell the agent HOW to behave and WHAT to do
    # We pass in our payload_fields so it knows what columns exist in our database
    instructions=f"""
    You analyze user queries and create SQL WHERE clauses for an e-commerce product database.
    Available columns: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any column you plan to filter on.
    Do NOT guess values — always check first.
    
    For example, if the user says "women's tops", call get_unique_values("index_name") 
    to find that women's products are under "Ladieswear", not "Women" or "Womens".
    
    CRITICAL RULES FOR YOUR RESPONSE:
    - Return ONLY a SQL WHERE clause (without the WHERE keyword) or None
    - Use %s as parameter placeholders
    - After the WHERE clause, on a new line starting with PARAMS:, list the parameter values as a JSON array
    - If no filters needed, return exactly: None
    
    GOOD response examples:
    - colour_group_name = %s
      PARAMS: ["Black"]
    - index_name = %s AND colour_group_name != %s
      PARAMS: ["Ladieswear", "Black"]
    - index_name = %s AND colour_group_name IN (%s, %s, %s)
      PARAMS: ["Menswear", "Dark Blue", "Dark Green", "Dark Grey"]
    
    BAD response (DO NOT DO THIS):
    - SELECT * FROM products WHERE ...
    - Any Python code or imports
    """,
    # This is where we give the agent access to our tool
    # The agent decides WHEN to call it based on the user's query
    tools=[Tool(get_unique_values)],
)

import json

# This function ties everything together:
# 1. Send the user's query to the agent
# 2. The agent reasons about what filters to create (calling tools as needed)
# 3. We parse the agent's WHERE clause and parameters
# 4. We embed the query and search CockroachDB with the generated filters
async def auto_filtered_search(user_query):
    # Step 1: Ask the agent to create filters — it may call get_unique_values behind the scenes
    filter_result = await smart_filter_agent.run(f"Create filters for: {user_query}")
    filter_output = filter_result.output.strip()

    print(f"Query: {user_query}")
    print(f"Generated filter: {filter_output}")

    # Step 2: Parse the WHERE clause and parameters
    where_clause = None
    filter_params = []
    if filter_output != "None":
        try:
            lines = filter_output.split("\n")
            where_clause = lines[0].strip()
            for line in lines[1:]:
                if line.strip().startswith("PARAMS:"):
                    filter_params = json.loads(line.strip().replace("PARAMS:", "").strip())
        except Exception as e:
            print(f"Filter parsing failed: {e}, searching without filters")
            where_clause = None
            filter_params = []

    # Step 3: Embed our query and search with the agent-generated filters
    query_vector = embed_text(user_query)
    where_sql = f"WHERE {where_clause}" if where_clause else ""
    params = filter_params + [str(query_vector)]

    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT prod_name, colour_group_name, section_name
            FROM products
            {where_sql}
            ORDER BY embedding <=> %s
            LIMIT 5
        """, params)
        results = cur.fetchall()

    # Step 4: Display what we found
    print(f"\nFound {len(results)} products:")
    for i, row in enumerate(results, 1):
        print(f"  {i}. {row[0]} ({row[1]}, {row[2]})")

    return results

Our agent has the tools it needs, lets ask it to find a garmet that fits specific properties.

In [ ]:
results = await auto_filtered_search("Show me white tops for women")

Now let's give it soem negative property filters (humans even struggle with these)

In [ ]:
results = await auto_filtered_search("Women's clothing but not black")

## Combining Query Optimization with Dynamic Filtering

Now let's build an agent that cleans up our messy human talk by optimizing the search query AND dynamically generates filters.

Natural language works well with semantic search, however, filler words and unnecessary context can reduce the accuracy of retrieval. LLMs understand vector databases and what words are not needed for retrieval, so we will let our agent create an optimized query for us with filters.

In [ ]:
from pydantic import BaseModel, Field

# Structured output — no more string parsing
class SearchPlan(BaseModel):
    optimized_query: str = Field(description="The optimized search query for vector similarity")
    where_clause: str | None = Field(description="SQL WHERE clause without the WHERE keyword, using %s placeholders. None if no filters needed.")
    params: list[str] = Field(description="Parameter values matching the %s placeholders in where_clause, in order. Empty list if no filters.")

# This agent does TWO things at once: optimizes the query AND generates filters
# Using output_type ensures the output is always valid structured data
combined_agent = Agent(
    model="openai:gpt-4.1-mini",
    output_type=SearchPlan,
    instructions=f"""
    You are an intelligent product search assistant that performs TWO tasks:
    
    1. OPTIMIZE the user's natural language query for better vector search results
    2. GENERATE appropriate SQL WHERE clauses based on the query context
    
    Available columns: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any column you plan to filter on.
    Do NOT guess values — always check first.
    
    IMPORTANT COLUMN USAGE:
    - For gender/category (men, women, kids): ALWAYS use "index_name" (e.g. "Menswear", "Ladieswear")
    - For color: use "colour_group_name"
    - For product type: use "product_type_name"
    - Do NOT use "department_name" unless the user specifically mentions a department
    
    Examples:
    - where_clause: "index_name = %s AND colour_group_name = %s", params: ["Menswear", "Dark Blue"]
    - where_clause: "colour_group_name NOT IN (%s, %s)", params: ["Black", "Dark Grey"]
    - where_clause: null, params: []
    """,
    tools=[Tool(get_unique_values)],
)

async def smart_search(user_query):
    print(f"Original query: '{user_query}'")
    print("=" * 70)
    
    # Step 1: Agent returns a validated SearchPlan object — no parsing needed
    agent_result = await combined_agent.run(f"Process this query: {user_query}")
    plan = agent_result.output
    
    print(f"Optimized query: '{plan.optimized_query}'")
    print(f"Generated filters: {plan.where_clause}")
    print(f"Params: {plan.params}")
    print("=" * 70)
    
    # Step 2: Embed the OPTIMIZED query and search with filters
    query_vector = embed_text(plan.optimized_query)
    vec_str = str(query_vector)
    
    if plan.where_clause:
        where_sql = f"WHERE {plan.where_clause}"
        params = [vec_str] + list(plan.params) + [vec_str]
    else:
        where_sql = ""
        params = [vec_str, vec_str]
    
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT prod_name, product_type_name, colour_group_name,
                   section_name, detail_desc, embedding <=> %s AS distance
            FROM products
            {where_sql}
            ORDER BY embedding <=> %s
            LIMIT 5
        """, params)
        results = cur.fetchall()
    
    # Step 3: Display results
    print(f"\nFound {len(results)} products:\n")
    for i, row in enumerate(results, 1):
        print(f"{i}. {row[0]}")
        print(f"   Type: {row[1]} | Color: {row[2]}")
        print(f"   Section: {row[3]} | Distance: {row[5]:.4f}")
        print(f"   {row[4][:120]}...")
        print()
    
    return results

### Example 1: Complex natural language query

In [ ]:
results = await smart_search(
    "I want to see men's dark blue sweaters"
)

### Example 2: Query with implicit filtering needs

In [ ]:
results = await smart_search(
    "I need something nice to wear to a wedding, maybe a dress, nothing too dark"
)

## Query Decomposition Agent

Now let's tackle multi-part queries. A question like "compare the men's jacket options with the women's coat options in dark colors" requires two separate retrieval calls with different filters.

In [ ]:
import json

decomposition_agent = Agent(
    model="openai:gpt-4.1-mini",
    instructions=f"""
    You decompose complex user queries into discrete subqueries for a product vector database.
    
    Available columns: {payload_fields}
    
    BEFORE generating any filter, you MUST use the get_unique_values tool to look up
    the actual values in the database for any column you plan to filter on.
    Do NOT guess values — always check first.
    
    For each subquery, return a JSON array where each element has:
    - "query": the optimized search query for this subquery
    - "where_clause": SQL WHERE clause without WHERE keyword, using %s placeholders (or null)
    - "params": JSON array of parameter values (or [])
    - "purpose": what this subquery is looking for
    
    Return ONLY valid JSON as your final answer. No explanation.
    """,
    tools=[Tool(get_unique_values)],
)

async def decomposed_search(user_query):
    print(f"Original query: '{user_query}'")
    print("=" * 70)
    
    result = await decomposition_agent.run(f"Decompose this query: {user_query}")
    subqueries = json.loads(result.output)
    
    print(f"Decomposed into {len(subqueries)} subqueries:\n")
    
    all_results = []
    for i, sq in enumerate(subqueries, 1):
        print(f"Subquery {i}: {sq['purpose']}")
        print(f"  Search: '{sq['query']}'")
        print(f"  Filters: {sq['where_clause']}")
        print(f"  Params: {sq.get('params', [])}")
        
        query_vector = embed_text(sq['query'])
        where_sql = f"WHERE {sq['where_clause']}" if sq.get('where_clause') else ""
        params = sq.get('params', []) + [str(query_vector)]
        
        with conn.cursor() as cur:
            cur.execute(f"""
                SELECT prod_name, colour_group_name, section_name
                FROM products
                {where_sql}
                ORDER BY embedding <=> %s
                LIMIT 3
            """, params)
            results = cur.fetchall()
        
        print(f"  Found {len(results)} results")
        for row in results:
            print(f"    - {row[0]} ({row[1]}, {row[2]})")
        print()
        
        all_results.extend(results)
    
    return all_results

In [ ]:
results = await decomposed_search(
    "Compare men's black sweaters with women's white tops"
)

## Key Takeaways

**What we've built:**
1. **Dynamic filtering agent** — Automatically generates SQL WHERE clauses from natural language
2. **Combined optimization + filtering** — Query optimization AND filter generation in one agent
3. **Query decomposition agent** — Breaks multi-part queries into targeted subqueries with individual filters

**The agent advantage:**
- **Before agents**: You'd need to manually write filter code, optimize queries, and chain operations
- **With agents**: Natural language in, optimized database queries with filters out — automatically

Next up: we take this from single-turn RAG to **conversational AI** with intent classification and multi-agent orchestration.

In [ ]:
conn.close()